In [ ]:
import pandas as pd

 Сколько отсекает каждый фильтр:

In [2]:
predf = pd.read_excel("../../data/interim/preclean_base.xlsx")

df = predf.copy()

conditions = {
    "birthday_mistake_ok": (df["birthday_mistake"].isna() | (df["birthday_mistake"] == 0)),
    "busy_type_full": (df["busy_type"] == "Полная занятость"),
    "country_rf": (df["country"] == "Российская Федерация"),
    "inner_info_status_approved": (df["inner_info_status"] == "Одобрено"),
    "experience_mistake_ok": (df["experience_mistake"] == 0),
    "inactive_eq_1": (df["inactive"] == 1),
    "relocation_ok": (df["relocation"].isna() | (df["relocation"] == 0)),
    "retraining_ok": (df["retraining_capability"].isna() | (df["retraining_capability"] == 1)),
    "business_trips_eq_0": (df["business_trips"] == 0),
    "gender_notna": df["gender"].notna(),
    "gender_not_empty": (df["gender"] != ""),
    "salary_gt_1000": (df["salary"] > 1000),
}

n_total = len(df)

rows = []
for name, cond in conditions.items():
    kept = int(cond.sum())
    dropped = n_total - kept
    rows.append({
        "filter": name,
        "kept_rows": kept,
        "dropped_rows": dropped,
        "dropped_pct": round(dropped / n_total * 100, 2)
    })

filter_impact_df = pd.DataFrame(rows).sort_values("dropped_rows", ascending=False)
print(filter_impact_df)

                        filter  kept_rows  dropped_rows  dropped_pct
9                 gender_notna      35048         14951        29.90
1               busy_type_full      46532          3467         6.93
8          business_trips_eq_0      46716          3283         6.57
5                inactive_eq_1      48394          1605         3.21
3   inner_info_status_approved      49467           532         1.06
6                relocation_ok      49514           485         0.97
11              salary_gt_1000      49576           423         0.85
7                retraining_ok      49680           319         0.64
2                   country_rf      49731           268         0.54
0          birthday_mistake_ok      49995             4         0.01
4        experience_mistake_ok      49995             4         0.01
10            gender_not_empty      49999             0         0.00


In [3]:
mask = (
    (df["birthday_mistake"].isna() | (df["birthday_mistake"] == 0)) &
    (df["busy_type"] == "Полная занятость") &
    (df["country"] == "Российская Федерация") &
    (df["inner_info_status"] == "Одобрено") &
    (df["experience_mistake"] == 0) &
    (df["inactive"] == 1) &
    (df["relocation"].isna() | df["relocation"] == 0) &
    (df["retraining_capability"].isna() | df["retraining_capability"] == 1) &
    (df["business_trips"] == 0) &
    (df["gender"].notna()) &
    (df["gender"] != "") &
    (df["salary"] > 1000)
)

final_cols = [
    "salary",
    "gender",
    "experience",
    "education_type",
    "region_code",
    "industry_code"
]

tmp_df = df.loc[mask, final_cols].copy()

print("После mask:", tmp_df.shape)

print("\nПропуски после mask:")
print(tmp_df.isna().sum())

print("\nСтроки без пропусков в final_cols:")
print(tmp_df.dropna(subset=final_cols).shape)

После mask: (28834, 6)

Пропуски после mask:
salary                0
gender                0
experience            0
education_type    22404
region_code           0
industry_code      3463
dtype: int64

Строки без пропусков в final_cols:
(5826, 6)


In [3]:
import pandas as pd

df = pd.read_excel('../../data/interim/preclean_base.xlsx')
total = len(df)
print(f'Всего записей в датасете: {total}\n')

filters = {
    'birthday_mistake == 1': df['birthday_mistake'] == 1,
    'experience_mistake == 1': df['experience_mistake'] == 1,
    'inactive != 1': df['inactive'] != 1,
    'salary <= 1000': df['salary'] <= 1000,
    'gender пустой': df['gender'] == '',
    'country != РФ': df['country'] != 'Российская Федерация',
    'busy_type != Полная занятость': df['busy_type'] != 'Полная занятость',
    'inner_info_status != Одобрено': df['inner_info_status'] != 'Одобрено',
    'business_trips != 0': df['business_trips'] != 0,
    'relocation != 0': df['relocation'] != 0,
    'retraining_capability != 1': df['retraining_capability'] != 1,
}

for name, condition in filters.items():
    count = condition.sum()
    pct = count / total * 100
    print(f'{name:<45} {count:>6} чел.  ({pct:.1f}%)')

print()
clean = df[
    (df['birthday_mistake'] != 1) &
    (df['experience_mistake'] != 1) &
    (df['inactive'] == 1) &
    (df['salary'] > 1000) &
    (df['gender'] != '') &
    (df['country'] == 'Российская Федерация') &
    (df['busy_type'] == 'Полная занятость') &
    (df['inner_info_status'] == 'Одобрено') &
    (df['business_trips'] == 0) &
    (df['retraining_capability'] == 1)
]

print(f'После всех фильтров осталось: {len(clean)} чел. ({len(clean)/total*100:.1f}% от исходных)')
print(f'Отсеяно: {total - len(clean)} чел. ({(total - len(clean))/total*100:.1f}%)')

Всего записей в датасете: 49999

birthday_mistake == 1                              4 чел.  (0.0%)
experience_mistake == 1                            4 чел.  (0.0%)
inactive != 1                                   1605 чел.  (3.2%)
salary <= 1000                                   423 чел.  (0.8%)
gender пустой                                      0 чел.  (0.0%)
country != РФ                                    268 чел.  (0.5%)
busy_type != Полная занятость                   3467 чел.  (6.9%)
inner_info_status != Одобрено                    532 чел.  (1.1%)
business_trips != 0                             3283 чел.  (6.6%)
relocation != 0                                 2898 чел.  (5.8%)
retraining_capability != 1                      4211 чел.  (8.4%)

После всех фильтров осталось: 39910 чел. (79.8% от исходных)
Отсеяно: 10089 чел. (20.2%)
